# HealthKit gold layer - ad hoc query notebook

Run the cell below once to open a connection. Auth is your local `az login`
session (`auth_type=azure-cli`) - no token or secret needed.

Then edit the query in the last cell and re-run it as many times as you
like; the connection stays open across cells.

**Kernel:** any Python environment with `pandas`, `sqlalchemy`, and
`databricks-sqlalchemy` installed works (see `requirements.txt`) - it
doesn't have to be `dbt/.venv` specifically.

In [ ]:
import os
import subprocess

# Some local network setups (e.g. a TLS-inspecting proxy) inject a CA cert
# that macOS trusts but Python's bundled `certifi` does not, which makes
# HTTPS calls to Databricks fail with SSLCertVerificationError. Build a
# combined bundle (certifi + macOS keychain) once and point Python at it.
# Harmless no-op if your network doesn't need it.
_ca_bundle = os.path.expanduser("~/.healthkit-ca-bundle.pem")
if not os.path.exists(_ca_bundle):
    import certifi

    with open(_ca_bundle, "w") as f:
        f.write(open(certifi.where()).read())
        for keychain in (
            "/Library/Keychains/System.keychain",
            "/System/Library/Keychains/SystemRootCertificates.keychain",
            os.path.expanduser("~/Library/Keychains/login.keychain-db"),
        ):
            result = subprocess.run(
                ["security", "find-certificate", "-a", "-p", keychain],
                capture_output=True,
                text=True,
            )
            f.write(result.stdout)

os.environ["SSL_CERT_FILE"] = _ca_bundle
os.environ["REQUESTS_CA_BUNDLE"] = _ca_bundle

In [ ]:
import pandas as pd
from sqlalchemy import create_engine, text

DATABRICKS_HOST = "adb-7405605320524740.0.azuredatabricks.net"
HTTP_PATH = "/sql/1.0/warehouses/997b45263de388bd"
CATALOG = "healthkit"
SCHEMA = "gold"

engine = create_engine(
    f"databricks://token:@{DATABRICKS_HOST}"
    f"?http_path={HTTP_PATH}&catalog={CATALOG}&schema={SCHEMA}",
    # enable_telemetry=False: skip the connector's anonymous usage beacon,
    # which otherwise retries noisily against the same CA-trust issue above.
    connect_args={"auth_type": "azure-cli", "enable_telemetry": False},
)
conn = engine.connect()
print("Connected.")

In [ ]:
def q(sql: str) -> pd.DataFrame:
    """Run a query on the open connection and return a DataFrame."""
    return pd.read_sql(text(sql), conn)

## Scratch query

Edit the SQL below and re-run this cell as much as you want - `conn` stays
open, no need to re-authenticate. Unqualified table names resolve against
`healthkit.gold`; use `healthkit.silver.<table>` to reach the silver layer.

In [ ]:
q("SELECT * FROM fct_weekly_trends ORDER BY week_start DESC LIMIT 10")